In [ ]:
def main(datasources, start_date, end_date):
    # BigQuant extracts main(), so all imports and local helpers stay inside it.
    import os
    import numpy as np
    import torch
    import transformer_train as tt

    def locate_model_file():
        candidates = [
            tt.MODEL_PATH,
            os.path.join(os.getcwd(), 'transformer_model.json'),
        ]
        for candidate in candidates:
            if os.path.isfile(candidate):
                return candidate
        raise FileNotFoundError(
            'transformer_model.json was not found; run the new transformer_train.py first'
        )

    def load_trained_model(device):
        checkpoint = tt.load_model(locate_model_file(), map_location=device)
        if checkpoint.get('format_version') != 4:
            raise RuntimeError(
                'The uploaded JSON is an old incompatible model; retrain with the new transformer_train.py'
            )
        if checkpoint.get('input_version') != tt.INPUT_VERSION:
            raise RuntimeError('input_version does not match transformer_train.py')
        if checkpoint.get('input_fields') != tt.INPUT_FIELDS:
            raise RuntimeError('input_fields do not match transformer_train.py')
        if checkpoint.get('input_transform') != tt.INPUT_TRANSFORM:
            raise RuntimeError('input preprocessing does not match transformer_train.py')

        model = tt.StockTransformer(**checkpoint['model_cfg']).to(device)
        model.load_state_dict(checkpoint['state_dict'], strict=True)
        parameter_count = tt.validate_model_size(model)
        if parameter_count != int(checkpoint.get('parameter_count', -1)):
            raise RuntimeError('model parameter metadata does not match the loaded weights')
        model.eval()
        stats = (
            np.asarray(checkpoint['mean'], dtype=np.float32),
            np.asarray(checkpoint['std'], dtype=np.float32),
        )
        if stats[0].shape != (len(tt.INPUT_FIELDS),):
            raise RuntimeError('normalization mean has an invalid shape')
        if stats[1].shape != (len(tt.INPUT_FIELDS),):
            raise RuntimeError('normalization std has an invalid shape')
        if not np.isfinite(stats[0]).all() or not np.isfinite(stats[1]).all():
            raise RuntimeError('normalization statistics contain non-finite values')
        if (stats[1] <= 0).any():
            raise RuntimeError('normalization std must be positive')
        return model, stats

    bar_table = datasources['bar1m']
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, stats = load_trained_model(device)
    members = tt.membership(start_date, end_date)
    instruments = members['instrument'].drop_duplicates().tolist()
    raw_scores = tt.predict_scores(
        model=model,
        table=bar_table,
        start_date=start_date,
        end_date=end_date,
        instruments=instruments,
        stats=stats,
        device=device,
    )
    return tt.compose_submission_scores(raw_scores, members)
